In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.nn import functional as F

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(device.type + " set")

cuda set


In [3]:
from torch.utils.data import TensorDataset, DataLoader

def read_idx3_images(path):
    with open(path, 'rb') as f:
        # skip the 16 byte header: magic number(4), count(4), rows(4), cols(4)
        data = np.fromfile(f, dtype=np.uint8, offset=16)
    # reshape to (#images, 784) 
    return data.reshape(-1, 784).astype(np.float32) / 255.0

def read_idx1_labels(path):
    with open(path, 'rb') as f:
        data = np.fromfile(f, dtype=np.uint8, offset=8)
    return data.astype(np.int64)

train_images = read_idx3_images('./micrograd/code/data/train-images.idx3-ubyte')
train_labels = read_idx1_labels('./micrograd/code/data/train-labels.idx1-ubyte')

x_train = torch.tensor(train_images, dtype=torch.float32)
y_train = torch.tensor(train_labels, dtype=torch.long)

train_ds = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=torch.Generator(device=device))


In [4]:
model = nn.Sequential(
    nn.Linear(784, 16),
    nn.GELU(),
    nn.Linear(16, 16),
    nn.GELU(),
    nn.Linear(16, 10)
)
model.to(device)

Sequential(
  (0): Linear(in_features=784, out_features=16, bias=True)
  (1): GELU(approximate='none')
  (2): Linear(in_features=16, out_features=16, bias=True)
  (3): GELU(approximate='none')
  (4): Linear(in_features=16, out_features=10, bias=True)
)

In [5]:
dummy_input = torch.randn(1, 784)
dummy_output = torch.randn(1, 10)

# feed forward.
output = model(dummy_input)

print(f"Output shape: {output.shape}") # [1, 10]
print(output)

Output shape: torch.Size([1, 10])
tensor([[-0.1546,  0.1510, -0.2256, -0.1635,  0.1995, -0.2186,  0.0763, -0.2856,
          0.1720, -0.1429]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [6]:
optimizer = optim.AdamW(model.parameters(), lr=0.001)
# training loop
for epoch in range(2):
    for images, labels in train_loader:
        outputs = model(images)
        loss = nn.CrossEntropyLoss()(outputs, labels)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} complete. Final batch loss: {loss.item():.4f}")

Epoch 1 complete. Final batch loss: 0.1437
Epoch 2 complete. Final batch loss: 0.2112


In [19]:
logits = model(x_train[0])
probs = F.softmax(logits, dim=0)
print(probs)
print(y_train[0])


tensor([3.6742e-03, 2.1671e-04, 3.7173e-03, 4.1401e-01, 1.2959e-07, 5.7613e-01,
        3.5135e-07, 9.6474e-04, 1.0843e-03, 2.0974e-04], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)
tensor(5, device='cuda:0')
